In [ ]:
!pip install transformers datasets seqeval --quiet

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
dataset = load_dataset("BramVanroy/conll2003")
print(dataset)

In [ ]:
pos_label_list = [
    '"', "''", '#', '$', '(', ')', ',', '.', ':', '``',
    'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS',
    'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$',
    'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG',
    'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'
]
num_labels = len(pos_label_list)
id2label = {i: l for i, l in enumerate(pos_label_list)}
label2id = {l: i for i, l in enumerate(pos_label_list)}
print(f"POS labels: {num_labels}")

In [ ]:
model_name = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer loaded.")

In [ ]:
def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=128,
        is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["pos_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev = None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(labels[wid])
            else:
                aligned.append(-100)
            prev = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

train_data = dataset["train"].shuffle(seed=42).select(range(2000))
val_data   = dataset["validation"].select(range(400))

train_tok = train_data.map(tokenize_and_align, batched=True, remove_columns=dataset["train"].column_names)
val_tok   = val_data.map(tokenize_and_align, batched=True, remove_columns=dataset["validation"].column_names)

print(f"Train: {len(train_tok)} | Val: {len(val_tok)}")

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("Model loaded.")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [pos_label_list[l] for l in row if l != -100]
        for row in labels
    ]
    true_preds = [
        [pos_label_list[p] for p, l in zip(pr, lr) if l != -100]
        for pr, lr in zip(predictions, labels)
    ]
    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-tiny-pos",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print("\nFinal Results:")
for k, v in results.items():
    print(f"  {k}: {round(v, 4)}")

In [ ]:
def predict_pos(sentence):
    words = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = tokenizer(words, is_split_into_words=True).word_ids()
    seen = set()
    print(f"\n{'Word':<22} {'POS Tag'}")
    print("-" * 34)
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            print(f"{words[wid]:<22} {id2label[preds[idx]]}")
            seen.add(wid)

predict_pos("The quick brown fox jumps over the lazy dog")
predict_pos("John has been working at a large company in New York")

In [ ]:
def predict_chunks(sentence):
    words = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    chunk_map = {
        'DT': 'B-NP',  'NN': 'B-NP',  'NNS': 'B-NP', 'NNP': 'B-NP', 'NNPS': 'B-NP',
        'JJ': 'B-NP',  'PRP': 'B-NP', 'PRP$': 'B-NP','CD': 'B-NP',
        'VB': 'B-VP',  'VBD': 'B-VP', 'VBG': 'B-VP', 'VBN': 'B-VP',
        'VBP': 'B-VP', 'VBZ': 'B-VP', 'MD': 'B-VP',
        'RB': 'B-ADVP','RBR': 'B-ADVP','RBS': 'B-ADVP',
        'IN': 'B-PP',  'TO': 'B-PP'
    }

    seen = set()
    print(f"\n{'Word':<22} {'POS':<10} {'Chunk'}")
    print("-" * 44)
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            pos = id2label[preds[idx]]
            print(f"{words[wid]:<22} {pos:<10} {chunk_map.get(pos, 'O')}")
            seen.add(wid)

predict_chunks("The large company announced strong quarterly results yesterday")
predict_chunks("She will be attending the international conference in Paris next week")

In [ ]:
model.save_pretrained("./bert-tiny-pos-final")
tokenizer.save_pretrained("./bert-tiny-pos-final")
print("Model saved.")